In [9]:
import os
import scanpy as sc
import anndata as ad
import pandas as pd
import time 
import numpy as np
import liana as li
import itertools
import re # spliting the data 
import scipy as sci
from scipy.stats import false_discovery_control
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
adata = sc.read_h5ad('/projects/bioinformatics/DB/scRNAseq_parkinson/dataset.h5ad')

In [4]:
adata

AnnData object with n_obs × n_vars = 2096155 × 17267
    obs: 'n_genes', 'n_counts', 'Brain_bank', 'RIN', 'path_braak_lb', 'derived_class2', 'PMI', 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'gene_name', 'n_cells', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'genome', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title', 'uid'
    obsm: 'X_umap'

In [4]:
lr_resource = pd.read_csv("/projects/bioinformatics/DB/CellCellCommunication/WithEnzymes/cpdb_cellchat_enz_liana_format.csv")

In [6]:
adata.obs[["donor_id", "disease"]].drop_duplicates()["disease"].value_counts()

disease
Parkinson disease    73
normal               24
Name: count, dtype: int64

In [10]:
# to check if all the doners corresponded to exactly one disease group
adata.obs[["donor_id", "disease"]].drop_duplicates().groupby("donor_id")["disease"].nunique().value_counts()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/4184139918.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


disease
1    97
Name: count, dtype: int64

In [15]:
# check whether some donors contributed far more cells/nuclei than others
cells_per_donor = (
    adata.obs
    .groupby(["disease", "donor_id"], observed=True)
    .size()
    .reset_index(name="n_cells")
)

cells_per_donor.groupby("disease")["n_cells"].describe()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/4121659574.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,count,mean,std,min,25%,50%,75%,max
disease,,,,,,,,
normal,24.0,18554.041667,9657.960346,423.0,12856.0,18565.0,24686.5,37638.0
Parkinson disease,73.0,22614.493151,8294.642610,1666.0,19142.0,23000.0,28947.0,38350.0


In [17]:
# donor representation for each cell typedonor representation for each cell type
adata.obs.groupby(
    ["disease", "cell_type"],
    observed=True
)["donor_id"].nunique()

disease            cell_type                        
normal             ependymal cell                       22
                   endothelial cell                     24
                   astrocyte                            24
                   oligodendrocyte                      24
                   GABAergic neuron                     24
                   glutamatergic neuron                 24
                   leukocyte                            23
                   central nervous system macrophage    24
                   oligodendrocyte precursor cell       24
                   mural cell                           24
                   central nervous system neuron        22
Parkinson disease  ependymal cell                       69
                   endothelial cell                     73
                   astrocyte                            73
                   oligodendrocyte                      73
                   GABAergic neuron                     73
   

In [18]:
# whether each donor has enough cells of each cell type
cell_counts = (
    adata.obs
    .groupby(["disease", "donor_id", "cell_type"], observed=True)
    .size()
    .reset_index(name="n_cells")
)

cell_counts.groupby(["disease", "cell_type"])["n_cells"].describe()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/3045897534.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


count          mean  \
disease           cell_type                                                
normal            ependymal cell                      22.0     91.909091   
                  endothelial cell                    24.0    103.583333   
                  astrocyte                           24.0   1684.500000   
                  oligodendrocyte                     24.0   9633.958333   
                  GABAergic neuron                    24.0   1473.666667   
                  glutamatergic neuron                24.0   3288.666667   
                  leukocyte                           23.0     27.478261   
                  central nervous system macrophage   24.0    916.708333   
                  oligodendrocyte precursor cell      24.0    999.666667   
                  mural cell                          24.0    177.375000   
                  central nervous system neuron       22.0    180.363636   
Parkinson disease ependymal cell                      69.0    152.710145   
                  endothelial cell                    73.0    117.589041   
                  astrocyte                           73.0   2561.972603   
                  oligodendrocyte                     73.0  11812.397260   
                  GABAergic neuron                    73.0   1603.082192   
                  glutamatergic neuron                73.0   3423.671233   
                  leukocyte                           71.0     24.352113   
                  central nervous system macrophage   73.0   1322.109589   
                  oligodendrocyte precursor cell      73.0   1213.602740   
                  mural cell                          73.0    186.410959   
                  central nervous system neuron       71.0    211.422535   

                                                             std    min  \
disease           cell_type                                               
normal            ependymal cell                      154.907625    1.0   
                  endothelial cell                     67.760394   12.0   
                  astrocyte                           849.275343  245.0   
                  oligodendrocyte                    5493.141110   47.0   
                  GABAergic neuron                    996.882343   14.0   
                  glutamatergic neuron               2574.266154   13.0   
                  leukocyte                            23.335244    1.0   
                  central nervous system macrophage   849.938539    1.0   
                  oligodendrocyte precursor cell      461.523627   72.0   
                  mural cell                          124.858812   12.0   
                  central nervous system neuron       226.385795    5.0   
Parkinson disease ependymal cell                      197.251788    1.0   
                  endothelial cell                     62.578137   30.0   
                  astrocyte                          1094.933445  372.0   
                  oligodendrocyte                    5129.648666  505.0   
                  GABAergic neuron                    801.984946   16.0   
                  glutamatergic neuron               2084.020214   10.0   
                  leukocyte                            15.967018    1.0   
                  central nervous system macrophage   727.529487   87.0   
                  oligodendrocyte precursor cell      393.842105  323.0   
                  mural cell                          106.809908   30.0   
                  central nervous system neuron       337.203231    1.0   

                                                         25%      50%  \
disease           cell_type                                             
normal            ependymal cell                        3.50     33.5   
                  endothelial cell                     51.75     94.5   
                  astrocyte                          1177.00   1654.0   
                  oligodendrocyte                    6421.00  

### Most cell types have plenty of cells per donor. However, three populations are relatively sparse:

Ependymal cells: median ~34 normal / ~100 PD
Leukocytes: median ~22 normal / ~20 PD
CNS neurons: median ~89 normal / ~87 PD

For the major populations—astrocytes, oligodendrocytes, GABAergic neurons, glutamatergic neurons, macrophages, OPCs, etc.—your donor-level representation looks strong.

In [20]:
adata.obs.columns.tolist()

['n_genes',
 'n_counts',
 'Brain_bank',
 'RIN',
 'path_braak_lb',
 'derived_class2',
 'PMI',
 'tissue_ontology_term_id',
 'tissue_type',
 'assay_ontology_term_id',
 'disease_ontology_term_id',
 'cell_type_ontology_term_id',
 'self_reported_ethnicity_ontology_term_id',
 'development_stage_ontology_term_id',
 'sex_ontology_term_id',
 'donor_id',
 'suspension_type',
 'is_primary_data',
 'cell_type',
 'assay',
 'disease',
 'sex',
 'tissue',
 'self_reported_ethnicity',
 'development_stage',
 'observation_joinid']

In [21]:
# how many cells you have for each donor × cell type × disease

adata.obs.groupby(
    ["disease", "donor_id", "cell_type"],
    observed=True
).size().describe()

count     1054.000000
mean      1988.761860
std       3613.152966
min          1.000000
25%         92.000000
50%        573.500000
75%       2068.500000
max      21558.000000
dtype: float64

This tells us that some donor × cell-type groups have extremely few cells: minimum = 1, while median = 574.

That matters because we don't want a donor with only 1–5 cells of a particular cell type to drive an LR interaction.

In [22]:
# see how many donor–cell-type groups have fewer than 20 cells.
cell_counts = (
    adata.obs
    .groupby(
        ["disease", "donor_id", "cell_type"],
        observed=True
    )
    .size()
    .reset_index(name="n_cells")
)

cell_counts["n_cells"].lt(20).sum()

np.int64(93)

93 out of 1,054 donor–cell-type groups have fewer than 20 cells (~8.8%). We shouldn't automatically remove them yet; we need to see which cell types they belong to.

In [23]:
cell_counts[cell_counts["n_cells"] < 20].groupby(
    ["disease", "cell_type"]
).size()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/3887453613.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


disease            cell_type                        
normal             ependymal cell                        9
                   endothelial cell                      3
                   astrocyte                             0
                   oligodendrocyte                       0
                   GABAergic neuron                      1
                   glutamatergic neuron                  1
                   leukocyte                            11
                   central nervous system macrophage     2
                   oligodendrocyte precursor cell        0
                   mural cell                            1
                   central nervous system neuron         5
Parkinson disease  ependymal cell                       14
                   endothelial cell                      0
                   astrocyte                             0
                   oligodendrocyte                       0
                   GABAergic neuron                      1
   

The sparse groups are concentrated mainly in ependymal cells, leukocytes, and CNS neurons, while the major populations are well supported.

dataset contains 5 tissues, and if PD and controls have different tissue composition, that can strongly affect cell–cell communication.

# check donors across brain regions

In [25]:
adata.obs.groupby(
    ["disease", "tissue"],
    observed=True
)["donor_id"].nunique()

disease            tissue                             
normal             dorsal motor nucleus of vagus nerve    19
                   medial globus pallidus                 19
                   primary motor cortex                   22
                   prefrontal cortex                      20
                   primary visual cortex                  21
Parkinson disease  dorsal motor nucleus of vagus nerve    70
                   medial globus pallidus                 70
                   primary motor cortex                   70
                   prefrontal cortex                      70
                   primary visual cortex                  63
Name: donor_id, dtype: int64

In [27]:
# check donor overlap across tissues
adata.obs.groupby(
    ["disease", "donor_id"],
    observed=True
)["tissue"].nunique().value_counts()

tissue
5    71
4    17
2     4
3     4
1     1
Name: count, dtype: int64

It means:

71 donors have all 5 brain regions.
17 donors have 4 regions.
Only 9 donors have 1–3 regions.

So your dataset has a strong within-donor repeated-measures structure. This is actually valuable because the same person often contributes multiple brain regions.

That means we should not simply pool all tissues together and pretend every donor is independent for every observation

In [29]:
# check which tissues are missing
tissue_by_donor = (
    adata.obs
    .groupby(["donor_id", "disease"], observed=True)["tissue"]
    .unique()
)

tissue_by_donor.head(20)

donor_id  disease          
CASE_53   Parkinson disease    ['dorsal motor nucleus of vagus nerve', 'prefr...
CASE_61   Parkinson disease    ['medial globus pallidus', 'dorsal motor nucle...
CASE_54   Parkinson disease    ['dorsal motor nucleus of vagus nerve', 'media...
CASE_67   Parkinson disease    ['primary visual cortex', 'primary motor corte...
CASE_69   Parkinson disease    ['primary motor cortex', 'prefrontal cortex', ...
CASE_55   Parkinson disease    ['medial globus pallidus', 'primary motor cort...
CTRL_18   normal               ['primary visual cortex', 'medial globus palli...
CASE_59   Parkinson disease    ['primary visual cortex', 'medial globus palli...
CASE_66   Parkinson disease    ['prefrontal cortex', 'medial globus pallidus'...
CASE_52   Parkinson disease    ['prefrontal cortex', 'primary visual cortex',...
CASE_71   Parkinson disease    ['medial globus pallidus', 'dorsal motor nucle...
CASE_58   Parkinson disease    ['primary motor cortex', 'dorsal motor nucleus...


This output gives us an important insight: many donors have multiple brain regions, so your dataset is not simply 97 independent samples. It has a repeated-measures structure.

In [31]:
tissue_by_donor.apply(len).groupby(
    tissue_by_donor.apply(len)
).size()

tissue
1     1
2     4
3     4
4    17
5    71
Name: tissue, dtype: int64

whether PD and control donors differ substantially in age. Age can strongly affect brain gene expression and potentially cell–cell communication.

In [32]:
adata.obs.groupby(
    ["disease", "donor_id"],
    observed=True
)["development_stage"].first().groupby("disease").describe()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/3750975935.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,count,unique,top,freq
disease,,,,
normal,24,16,66-year-old stage,3
Parkinson disease,73,28,81-year-old stage,7


Normal: 24 donors, 16 unique ages; most frequent age = 66
PD: 73 donors, 28 unique ages; most frequent age = 81

In [33]:
age_by_donor = (
    adata.obs[["donor_id", "disease", "development_stage"]]
    .drop_duplicates("donor_id")
    .copy()
)

age_by_donor["age"] = (
    age_by_donor["development_stage"]
    .str.extract(r"(\d+)")
    .astype(float)
)

age_by_donor.groupby("disease")["age"].describe()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/2693667290.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,count,mean,std,min,25%,50%,75%,max
disease,,,,,,,,
normal,24.0,75.916667,10.235984,64.0,67.0,74.0,80.0,101.0
Parkinson disease,73.0,79.369863,7.430244,57.0,75.0,80.0,84.0,102.0


Now we know age is potentially important, although the difference isn't enormous:

Normal: median 74 years
PD: median 80 years
Normal: 24 donors
PD: 73 donors

So age should probably be considered as a covariate in the eventual analysis rather than ignoring it.

In [34]:
# sex balance 

sex_by_donor = (
    adata.obs[["donor_id", "disease", "sex"]]
    .drop_duplicates("donor_id")
)

pd.crosstab(
    sex_by_donor["disease"],
    sex_by_donor["sex"]
)

sex,female,male
disease,,
normal,9,15
Parkinson disease,28,45


The proportions are almost identical:

Normal: 37.5% female
PD: 38.4% female

So sex does not look like a major confounder here.

# Brain Bank

samples from different banks can have technical/biological differences.

In [35]:
brainbank_by_donor = (
    adata.obs[["donor_id", "disease", "Brain_bank"]]
    .drop_duplicates("donor_id")
)

pd.crosstab(
    brainbank_by_donor["disease"],
    brainbank_by_donor["Brain_bank"]
)

Brain_bank,Harvard,MSSM,UD,UM
disease,,,,
normal,3,12,0,9
Parkinson disease,22,12,21,18


The biggest issue is UD: you have 21 PD donors but 0 controls. Harvard is also heavily PD-skewed.

That means Brain Bank is confounded with disease. If we see an LR difference between PD and normal, some of that difference could potentially be related to the source/brain bank rather than PD itself.

This makes me less enthusiastic about a simple Mann–Whitney test as the main analysis. We may still use it as an exploratory analysis, but for a stronger result we should eventually consider donor-level modeling with covariates.

# check RIN

RIN (RNA Integrity Number) measures the quality/integrity of the RNA. Poorer-quality samples can affect gene expression measurements.

In [36]:
rin_by_donor = (
    adata.obs[["donor_id", "disease", "RIN"]]
    .drop_duplicates("donor_id")
)

rin_by_donor.groupby("disease")["RIN"].describe()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/1292258305.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,count,mean,std,min,25%,50%,75%,max
disease,,,,,,,,
normal,24.0,3.71250,1.34853,1.2,2.75,4.0,4.55,6.3
Parkinson disease,73.0,3.80411,1.04502,1.4,3.20,3.8,4.60,6.0


RIN looks fairly similar between groups, so it doesn't appear to be a major disease-confounding variable:

Normal median: 4.0
PD median: 3.8
Normal mean: 3.71
PD mean: 3.80

There is some variability, but the distributions are broadly comparable.

# PMI
= post-mortem interval, the time between death and tissue preservation. It can affect RNA quality and gene expression.

In [37]:
pmi_by_donor = (
    adata.obs[["donor_id", "disease", "PMI"]]
    .drop_duplicates("donor_id")
)

pmi_by_donor.groupby("disease")["PMI"].describe()

/var/tmp/pbs.441868.pbs01/ipykernel_292420/1792219544.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,count,mean,std,min,25%,50%,75%,max
disease,,,,,,,,
normal,23.0,16.614638,5.939681,4.550000,12.625,19.0,21.00,26.0
Parkinson disease,73.0,15.458219,7.145856,3.833333,8.500,19.0,20.91,30.0


PMI also looks reasonably comparable:

Normal median: 19 h
PD median: 19 h
Normal mean: 16.6 h
PD mean: 15.5 h
One normal donor has missing PMI.

So PMI doesn't currently look like a major confounder.